In [1]:
!apt-get install openjdk-17-jdk-headless -qq > /dev/null
!pip install pyspark nltk -qq

import os
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'

In [2]:
!pip install gdown -q
import gdown

CATEGORY_NAME = 'Kindle_Store'

FILE_ID = '1AhlTkSclHtP8IL7LeON0FCcXRln7JvlH'
INPUT_PATH = f'/content/{CATEGORY_NAME}.jsonl.gz'

gdown.download(id=FILE_ID, output=INPUT_PATH, quiet=False)

OUTPUT_DIR = '/content/outputs'
OUTPUT_PARQUET = f'{OUTPUT_DIR}/cleaned_{CATEGORY_NAME}.parquet'
REPORT_PATH = f'{OUTPUT_DIR}/reports/reviews_cleaning1_report_{CATEGORY_NAME}.txt'

Downloading...
From (original): https://drive.google.com/uc?id=1AhlTkSclHtP8IL7LeON0FCcXRln7JvlH
From (redirected): https://drive.google.com/uc?id=1AhlTkSclHtP8IL7LeON0FCcXRln7JvlH&confirm=t&uuid=191c6eea-a8d7-4da7-87d3-c33951473225
To: /content/Kindle_Store.jsonl.gz
100%|██████████| 4.57G/4.57G [00:37<00:00, 120MB/s]


In [15]:
import gzip

EXAMPLE_LINES = 100
EXAMPLE_PATH = '/content/outputs/example_data.jsonl'

with gzip.open(INPUT_PATH, 'rt', encoding='utf-8') as fin, \
     open(EXAMPLE_PATH, 'w', encoding='utf-8') as fout:
    for i, line in enumerate(fin):
        if i >= EXAMPLE_LINES:
            break
        fout.write(line)



In [3]:
from pyspark.sql import SparkSession, functions as F, Window
from pyspark.sql.types import StringType, ArrayType

spark = SparkSession.builder \
    .appName('Kindle Review Cleaning') \
    .master('local[*]') \
    .config('spark.driver.memory', '40g') \
    .config('spark.driver.maxResultSize', '8g') \
    .config('spark.sql.shuffle.partitions', '200') \
    .config('spark.sql.adaptive.enabled', 'true') \
    .config('spark.sql.adaptive.coalescePartitions.enabled', 'true') \
    .config('spark.serializer', 'org.apache.spark.serializer.KryoSerializer') \
    .config('spark.sql.parquet.compression.codec', 'gzip') \
    .getOrCreate()

spark.sparkContext.setLogLevel('WARN')

In [4]:
input_size_bytes = os.path.getsize(INPUT_PATH)

df_raw = spark.read.json(INPUT_PATH) \
    .select('parent_asin', 'title', 'text', 'rating')

df_raw = df_raw.withColumn('review_id', F.monotonically_increasing_id() + 1)
df_raw.cache()

initial_count = df_raw.count()
print(f'Số review ban đầu: {initial_count:,}')
df_raw.show(5, truncate=50)

Số review ban đầu: 25,577,616
+-----------+--------------------------------------------------+--------------------------------------------------+------+---------+
|parent_asin|                                             title|                                              text|rating|review_id|
+-----------+--------------------------------------------------+--------------------------------------------------+------+---------+
| B00LXRJICK|      excellent writer reminds me of Clive Cussler|GRUMLEY is on par with Clive Cussler and his Di...|   5.0|        1|
| B073DFP8VC|                                      Alright book|The book Fade was not my favorite but was a goo...|   3.0|        2|
| B07QVH25KX|Hats off to Fern Michaels for all her great acc...|I have been a fan of this author for many years...|   5.0|        3|
| B004Y1NWQU|                                        Great read|This book is more than just about a dog and a m...|   5.0|        4|
| B08M993CNC|                          

In [5]:
df = df_raw.withColumn('title', F.coalesce(F.trim('title'), F.lit(''))) \
           .withColumn('text', F.coalesce(F.trim('text'), F.lit('')))

df = df.withColumn('review_text',
    F.when((F.col('title') != '') & (F.col('text') != ''),
           F.concat(F.col('title'), F.lit('. '), F.col('text')))
     .when(F.col('title') != '', F.col('title'))
     .otherwise(F.col('text'))
).drop('title', 'text')

print('Gộp title + text thành review_text')
df.show(5, truncate=80)

Gộp title + text thành review_text
+-----------+------+---------+--------------------------------------------------------------------------------+
|parent_asin|rating|review_id|                                                                     review_text|
+-----------+------+---------+--------------------------------------------------------------------------------+
| B00LXRJICK|   5.0|        1|excellent writer reminds me of Clive Cussler. GRUMLEY is on par with Clive Cu...|
| B073DFP8VC|   3.0|        2|Alright book. The book Fade was not my favorite but was a good read. I am a p...|
| B07QVH25KX|   5.0|        3|Hats off to Fern Michaels for all her great accomplishments❗. I have been a f...|
| B004Y1NWQU|   5.0|        4|Great read. This book is more than just about a dog and a man who is blind so...|
| B08M993CNC|   5.0|        5|Add to legend f Arthur. Good twist on the ledgen of King<br />Arthur ! Most i...|
+-----------+------+---------+---------------------------------------

In [6]:
df = df.filter(
    (F.col('review_text').isNotNull()) &
    (F.trim('review_text') != '')
)

after_empty = df.cache().count()
print(f'Xóa {initial_count - after_empty:,} review rỗng. Còn {after_empty:,}')

Xóa 0 review rỗng. Còn 25,577,616


In [7]:
df_raw.unpersist()

df = df.dropDuplicates(['parent_asin', 'review_text'])

after_dedup = df.cache().count()
print(f'Xóa {after_empty - after_dedup:,} review trùng. Còn lại {after_dedup:,} review ')

Xóa 315,024 review trùng. Còn lại 25,262,592 review 


In [8]:
@F.udf(StringType())
def clean_text(text):
    if not text:
        return text
    import re, unicodedata
    text = re.sub(r'<[^>]+>', ' ', text)
    text = unicodedata.normalize('NFKC', text)
    text = re.sub(r'[^\w\s.,!?;:\'\"()\-/]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df.unpersist()
df = df.withColumn('review_text', clean_text('review_text'))

df = df.filter(
    (F.col('review_text').isNotNull()) &
    (F.trim('review_text') != '')
)

after_clean = df.cache().count()
print(f'Chuẩn hóa xong. Còn {after_clean:,} review')
df.show(5, truncate=80)

Chuẩn hóa xong. Còn 25,262,592 review
+-----------+------+---------+--------------------------------------------------------------------------------+
|parent_asin|rating|review_id|                                                                     review_text|
+-----------+------+---------+--------------------------------------------------------------------------------+
| B004DI7HZ6|   5.0|      207|A great story!. One of my favorite books. I love the story, the writing is ve...|
| B0053TMP24|   4.0|      246|HEROES. I'm an adult reader and have enjoyed the Percy Jackson series. I'm gl...|
| B07TKJC2SC|   5.0|      279|Wow..awesome story!. I just finished this book and I have to leave my review ...|
| B00R5GJM6U|   5.0|      446|Yes!!!. After living in the South for ten years and then moving back to my be...|
| B06XQ85LY5|   5.0|      460|Five well earned stars.... Loved book two. Levy grew as a character. I learne...|
+-----------+------+---------+------------------------------------

In [9]:
import nltk
nltk.download('punkt_tab', quiet=True)

@F.udf(ArrayType(StringType()))
def split_sentences(text):
    from nltk.tokenize import sent_tokenize
    if not text:
        return []
    sentences = sent_tokenize(text)
    return [s.strip() for s in sentences if s.strip()]

df_sentences = df.withColumn('sentences', split_sentences('review_text')) \
    .select(
        'parent_asin', 'review_id', 'rating',
        F.posexplode('sentences').alias('_pos', 'sentence_text')
    ) \
    .withColumn('sentence_id', F.col('_pos') + 1) \
    .drop('_pos')

df.unpersist()

df_final = df_sentences.select('parent_asin', 'review_id', 'sentence_id', 'sentence_text', 'rating')
df_final.cache()

sentence_count = df_final.count()
print(f'Tách thành {sentence_count:,} câu')
df_final.show(10, truncate=80)

Tách thành 146,902,993 câu
+-----------+---------+-----------+--------------------------------------------------------------------------------+------+
|parent_asin|review_id|sentence_id|                                                                   sentence_text|rating|
+-----------+---------+-----------+--------------------------------------------------------------------------------+------+
| B004DI7HZ6|      207|          1|                                                                 A great story!.|   5.0|
| B004DI7HZ6|      207|          2|                                                       One of my favorite books.|   5.0|
| B004DI7HZ6|      207|          3|             I love the story, the writing is very good, i couldn't put it down!|   5.0|
| B004DI7HZ6|      207|          4|                                           I would highly recommend this series!|   5.0|
| B0053TMP24|      246|          1|                                                                      

In [ ]:
os.makedirs(os.path.dirname(OUTPUT_PARQUET), exist_ok=True)

df_final.write.mode('overwrite').parquet(OUTPUT_PARQUET)

In [ ]:
def get_dir_size(path):
    total = 0
    for dirpath, dirnames, filenames in os.walk(path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            total += os.path.getsize(fp)
    return total

output_size_bytes = get_dir_size(OUTPUT_PARQUET)

def format_size(size_bytes):
    if size_bytes >= 1024**3:
        return f'{size_bytes / 1024**3:.2f}GB'
    elif size_bytes >= 1024**2:
        return f'{size_bytes / 1024**2:.2f}MB'
    else:
        return f'{size_bytes / 1024:.2f}KB'

input_real_bytes = df_raw.select(
    F.sum(F.length(F.col('title')) + F.length(F.col('text')))
).collect()[0][0]

output_size_bytes = get_dir_size(OUTPUT_PARQUET)

input_size_str = format_size(input_real_bytes)
output_size_str = format_size(output_size_bytes)

In [13]:
reduction_pct = (1 - after_clean / initial_count) * 100
size_reduction_pct = (1 - output_size_bytes / input_real_bytes) * 100

report = f"""Kết quả đầu ra: {CATEGORY_NAME}:

Số review ban đầu:       {initial_count:,}
Số review còn lại:       {after_clean:,}
Số câu sau khi tách:     {sentence_count:,}
Kích thước ban đầu:      {input_size_str}
Kích thước cuối cùng:    {output_size_str}
"""

os.makedirs(os.path.dirname(REPORT_PATH), exist_ok=True)
with open(REPORT_PATH, 'w', encoding='utf-8') as f:
    f.write(report)

print(report)


Kết quả đầu ra: Kindle_Store:

Số review ban đầu:       25,577,616
Số review còn lại:       25,262,592
Số câu sau khi tách:     146,902,993
Kích thước ban đầu:      9.29GB
Kích thước cuối cùng:    4.19GB



In [14]:
!zip -r /content/output_data.zip /content/outputs

from google.colab import files
files.download('/content/output_data.zip')

  adding: content/outputs/ (stored 0%)
  adding: content/outputs/reports/ (stored 0%)
  adding: content/outputs/reports/reviews_cleaning1_report_Kindle_Store.txt (deflated 28%)
  adding: content/outputs/cleaned_Kindle_Store.parquet/ (stored 0%)
  adding: content/outputs/cleaned_Kindle_Store.parquet/.part-00003-7d39938a-da3d-4a31-8b96-784d56e6e1cd-c000.gz.parquet.crc (deflated 0%)
  adding: content/outputs/cleaned_Kindle_Store.parquet/part-00103-7d39938a-da3d-4a31-8b96-784d56e6e1cd-c000.gz.parquet (deflated 0%)
  adding: content/outputs/cleaned_Kindle_Store.parquet/part-00141-7d39938a-da3d-4a31-8b96-784d56e6e1cd-c000.gz.parquet (deflated 0%)
  adding: content/outputs/cleaned_Kindle_Store.parquet/part-00139-7d39938a-da3d-4a31-8b96-784d56e6e1cd-c000.gz.parquet (deflated 0%)
  adding: content/outputs/cleaned_Kindle_Store.parquet/part-00058-7d39938a-da3d-4a31-8b96-784d56e6e1cd-c000.gz.parquet (deflated 0%)
  adding: content/outputs/cleaned_Kindle_Store.parquet/part-00113-7d39938a-da3d-4a31-

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>